In [ ]:
import sys
from pathlib import Path

root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file()),
    None,
)
if root and str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from py_entry.strategy_hub import (
    build_strategy_runtime,
    get_strategy_refs,
)


strategy_refs = get_strategy_refs()
print("可用策略:", [i for i in strategy_refs if i.startswith("search:")])

STRATEGY = strategy_refs[0]
# STRATEGY = "search:macd_plain.macd_plain"
STRATEGY = "search:adx_macd_mtf.adx_macd_mtf"
RUN_SYMBOL = "DOGE/USDT"
print("当前策略:", STRATEGY)
print("运行品种:", RUN_SYMBOL)

cfg, stage_cfgs, bt = build_strategy_runtime(
    STRATEGY,
    run_symbol=RUN_SYMBOL,
)

In [ ]:
# WF fail-fast 已统一收口到 Rust 正式入口
print("WF 将直接通过 bt.walk_forward(stage_cfgs['wf_cfg']) 触发正式校验")

In [ ]:
# 阶段1：回测
backtest_view = bt.run()
backtest_view.print_report()

In [ ]:
from py_entry.io import DisplayConfig, DashboardOverride
from py_entry.runner import FormatResultsConfig

config = DisplayConfig(
    embed_data=False,
    width="100%",
    aspect_ratio="16/9",
    override=DashboardOverride(
        show=["0,0,0,1"],
        showInLegend=["0,0,0,1"],
        showRiskLegend="1,1,1,1",
        showLegendInAll=True,
    ).to_dict(),
)

backtest_bundle = backtest_view.prepare_export(
    FormatResultsConfig(dataframe_format="parquet", indicator_layout=cfg.chart_layout)
)
backtest_bundle.display(config=config)

In [ ]:
# 阶段2：优化
optimization_view = bt.optimize(stage_cfgs["opt_cfg"])
optimization_view.print_report()

In [ ]:
# 阶段3：参数抖动
sensitivity_view = bt.sensitivity(stage_cfgs["sens_cfg"])
sensitivity_view.print_report()

In [ ]:
# 阶段4：向前测试
walk_forward_view = bt.walk_forward(stage_cfgs["wf_cfg"])
walk_forward_view.print_report()

In [ ]:
# WF stitched 不复用含 paramKey 的单次回测布局；多窗口参数没有唯一阈值解释。
walk_forward_bundle = walk_forward_view.prepare_export(
    FormatResultsConfig(dataframe_format="parquet")
)
walk_forward_bundle.display(config=config)